# 03 — Gold Layer: Modelo Analítico para BI

Constrói tabelas finais desnormalizadas, orientadas a análise, com nomes claros e prontas para consumo direto pelo Analista de BI.

```
gold.dim_clientes        ← silver.dim_clientes
gold.dim_produtos        ← silver.dim_produtos
gold.dim_vendedores      ← silver.dim_vendedores + dim_canais + dim_regioes
gold.fct_pedidos         ← silver.fct_pedidos_cabecalho + clientes + vendedores
gold.fct_itens_pedido    ← silver.fct_pedidos_itens + produtos + contexto pedido
gold.fct_entregas        ← silver.fct_entregas + promised_date do pedido
gold.fct_ocorrencias     ← silver.fct_ocorrencias + contexto pedido
gold.mart_desempenho_mensal ← fct_pedidos agregado por período/canal/região/segmento
```

## Setup

In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG sandbox")
spark.sql("CREATE DATABASE IF NOT EXISTS gold COMMENT 'Camada Gold — tabelas analíticas prontas para BI'")
print("sandbox.gold pronto")

sandbox.gold pronto


## 1. gold.dim_clientes

In [0]:
df_cli = spark.table("silver.dim_clientes")

df_gold_clientes = df_cli.select(
    F.col("customer_id").alias("id_cliente"),
    F.col("nome_cliente"),
    F.col("segmento"),
    F.col("porte"),
    F.col("cidade"),
    F.col("estado"),
    F.col("data_cadastro"),
    F.year("data_cadastro").alias("ano_cadastro"),
    F.col("email"),
    F.col("status_cliente"),
    F.col("data_cadastro_flag_invalida").alias("flag_data_invalida"),
)

df_gold_clientes.write.format("delta").mode("overwrite").saveAsTable("gold.dim_clientes")
print(f"gold.dim_clientes: {df_gold_clientes.count()} registros")

gold.dim_clientes: 183 registros


## 2. gold.dim_produtos

In [0]:
df_prod = spark.table("silver.dim_produtos")

df_gold_produtos = df_prod.select(
    F.col("product_id").alias("id_produto"),
    F.col("product_name").alias("nome_produto"),
    F.col("category").alias("categoria"),
    F.col("subcategory").alias("subcategoria"),
    F.col("family").alias("familia"),
    F.col("status_produto"),
    F.col("list_price").alias("preco_lista"),
    F.col("currency").alias("moeda"),
    F.col("tags"),
)

df_gold_produtos.write.format("delta").mode("overwrite").saveAsTable("gold.dim_produtos")
print(f"gold.dim_produtos: {df_gold_produtos.count()} registros")

gold.dim_produtos: 71 registros


## 3. gold.dim_vendedores

Vendedores com canal e região já resolvidos — o analista não precisa fazer joins adicionais para segmentar por canal ou região.

In [0]:
df_vend = spark.table("silver.dim_vendedores")
df_can  = spark.table("silver.dim_canais")
df_reg  = spark.table("silver.dim_regioes")

df_gold_vendedores = (
    df_vend
    .join(df_can.select("id_canal", "nome_canal", "tipo_canal"),
          df_vend.canal_id == df_can.id_canal, "left")
    .join(df_reg.select(
              df_reg.regional_code.alias("reg_code"),
              df_reg.regional_name
          ),
          df_vend.regional_code == F.col("reg_code"), "left")
    .select(
        df_vend.seller_id.alias("id_vendedor"),
        df_vend.seller_name.alias("nome_vendedor"),
        df_vend.canal_id,
        df_can.nome_canal,
        df_can.tipo_canal,
        df_vend.regional_code,
        F.col("regional_name").alias("nome_regional"),
        df_vend.hire_date.alias("data_admissao"),
        df_vend.status.alias("status_vendedor"),
        df_vend.flag_canal_invalido,
    )
)

df_gold_vendedores.write.format("delta").mode("overwrite").saveAsTable("gold.dim_vendedores")
print(f"gold.dim_vendedores: {df_gold_vendedores.count()} registros")
df_gold_vendedores.show(truncate=False)

gold.dim_vendedores: 40 registros
+-----------+-------------+--------+------------+----------+-------------+-------------+-------------+---------------+-------------------+
|id_vendedor|nome_vendedor|canal_id|nome_canal  |tipo_canal|regional_code|nome_regional|data_admissao|status_vendedor|flag_canal_invalido|
+-----------+-------------+--------+------------+----------+-------------+-------------+-------------+---------------+-------------------+
|V001       |Vendedor 1   |CH01    |Inside Sales|DIRETO    |S            |Sul          |2024-06-27   |Desconhecido   |false              |
|V002       |Vendedor 2   |CH04    |Field Sales |DIRETO    |S            |Sul          |2023-12-16   |Ativo          |false              |
|V003       |Vendedor 3   |CH03    |Marketplace |DIGITAL   |S            |Sul          |2023-11-29   |Ativo          |false              |
|V004       |Vendedor 4   |CH02    |Parceiros   |INDIRETO  |NE           |Nordeste     |2025-02-11   |Ativo          |false         

##4. gold.dim_canais

Granularidade: 1 registro por canal de venda. Dimensão standalone para filtros e drill-down por canal no BI.

In [0]:
# gold.dim_canais
df_can = spark.table("silver.dim_canais")

df_gold_canais = df_can.select(
    F.col("id_canal"),
    F.col("nome_canal"),
    F.col("tipo_canal"),
    F.col("ativo_flag").alias("ativo"),
    F.col("observacao"),
)

df_gold_canais.write.format("delta").mode("overwrite").saveAsTable("gold.dim_canais")
print(f"gold.dim_canais: {df_gold_canais.count()} registros")
df_gold_canais.show(truncate=False)

gold.dim_canais: 7 registros
+--------+------------+----------+-----+---------------+
|id_canal|nome_canal  |tipo_canal|ativo|observacao     |
+--------+------------+----------+-----+---------------+
|CH01    |Inside Sales|DIRETO    |true |NULL           |
|CH02    |Parceiros   |INDIRETO  |true |NULL           |
|CH03    |Marketplace |DIGITAL   |true |NULL           |
|CH04    |Field Sales |DIRETO    |false|canal legado   |
|CH05    |E-commerce  |DIGITAL   |true |NULL           |
|CH06    |NOME_AUSENTE|INDIRETO  |true |nome ausente   |
|CH07    |Revendas    |INDIRETO  |NULL |id em lowercase|
+--------+------------+----------+-----+---------------+



##5. gold.dim_regioes
Granularidade: 1 registro por região. Dimensão standalone com gestor responsável para análises regionais.

In [0]:
# gold.dim_regioes
df_reg = spark.table("silver.dim_regioes")

df_gold_regioes = df_reg.select(
    F.col("regional_code").alias("id_regional"),
    F.col("regional_name").alias("nome_regional"),
    F.col("state").alias("estado_sede"),
    F.col("manager_name").alias("gestor"),
)

df_gold_regioes.write.format("delta").mode("overwrite").saveAsTable("gold.dim_regioes")
print(f"gold.dim_regioes: {df_gold_regioes.count()} registros")
df_gold_regioes.show(truncate=False)

gold.dim_regioes: 6 registros
+-----------+-------------+------------+--------------+
|id_regional|nome_regional|estado_sede |gestor        |
+-----------+-------------+------------+--------------+
|CO         |Centro-Oeste |GO          |Paulo Teixeira|
|N          |Norte        |AM          |Mariana Lopes |
|NE         |Nordeste     |BA          |Carlos Lima   |
|S          |Sul          |SC          |Rafael Souza  |
|SE         |Sudeste      |SP          |Ana Costa     |
|SUL        |Região Sul   |STA CATARINA|Rafael Souza  |
+-----------+-------------+------------+--------------+



## 6. gold.fct_pedidos

Granularidade: 1 registro por pedido. Enriquecido com canal, região, segmento e porte do cliente — pronto para KPIs diretos sem joins.

In [0]:
df_ped  = spark.table("silver.fct_pedidos_cabecalho")
df_cli  = spark.table("silver.dim_clientes")
df_vend = spark.table("silver.dim_vendedores")
df_can  = spark.table("silver.dim_canais")
df_reg  = spark.table("silver.dim_regioes")

df_fct_pedidos = (
    df_ped
    .join(df_cli.select("customer_id","segmento","porte","estado","status_cliente"),
          df_ped.customer_code == df_cli.customer_id, "left")
    .join(df_vend.select(
              F.col("seller_id").alias("v_seller_id"),
              F.col("canal_id").alias("v_canal_id"),
              F.col("regional_code").alias("v_regional_code")
          ),
          df_ped.seller_id == F.col("v_seller_id"), "left")
    .join(df_can.select("id_canal","nome_canal","tipo_canal"),
          F.col("v_canal_id") == df_can.id_canal, "left")
    .join(df_reg.select("regional_code","regional_name"),
          F.col("v_regional_code") == df_reg.regional_code, "left")
    .withColumn("flag_cancelado",    F.col("status_order") == "CANCELADO")
    .withColumn("flag_entregue",     F.col("status_order") == "ENTREGUE")
    .withColumn("flag_faturado",     F.col("status_order") == "FATURADO")
    .withColumn("flag_em_separacao", F.col("status_order") == "EM SEPARACAO")
    .withColumn("mes_pedido",  F.date_format("order_date", "yyyy-MM"))
    .withColumn("ano_pedido",  F.year("order_date"))
    .withColumn("mes_num",     F.month("order_date"))
    .withColumn("trimestre",   F.quarter("order_date"))
    .withColumn("pct_desconto",
        F.when(F.col("gross_amount") > 0,
               F.round(F.col("discount_amount") / F.col("gross_amount") * 100, 2))
         .otherwise(None)
    )
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_code").alias("id_cliente"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("order_date").alias("data_pedido"),
        F.col("promised_date").alias("data_prometida"),
        "mes_pedido", "ano_pedido", "mes_num", "trimestre",
        F.col("status_order").alias("status_pedido"),
        "flag_status_nulo", "flag_cancelado", "flag_entregue",
        "flag_faturado", "flag_em_separacao",
        F.col("gross_amount").alias("receita_bruta"),
        F.col("discount_amount").alias("desconto"),
        F.col("net_amount").alias("receita_liquida"),
        "pct_desconto",
        "qtd_itens", "qtd_produtos_distintos",
        F.col("v_canal_id").alias("canal_id"),
        "nome_canal", "tipo_canal",
        F.col("v_regional_code").alias("regional_code"),
        "regional_name",
        F.col("segmento").alias("segmento_cliente"),
        F.col("porte").alias("porte_cliente"),
        F.col("estado").alias("estado_cliente"),
        "payment_source", "payment_priority",
    )
)

df_fct_pedidos.write.format("delta").mode("overwrite").saveAsTable("gold.fct_pedidos")
print(f"gold.fct_pedidos: {df_fct_pedidos.count()} registros")

gold.fct_pedidos: 413 registros


## 7. gold.fct_itens_pedido

Granularidade: 1 registro por item. Enriquecido com produto e contexto do pedido (canal, região, segmento).

In [0]:
df_itens    = spark.table("silver.fct_pedidos_itens")
df_prod     = spark.table("silver.dim_produtos")
df_ped_gold = spark.table("gold.fct_pedidos")

df_fct_itens = (
    df_itens
    .join(
        df_prod.select(
            df_prod.product_id,
            df_prod.product_name.alias("nome_produto"),
            df_prod.category.alias("categoria"),
            df_prod.subcategory.alias("subcategoria"),
            df_prod.family.alias("familia"),
            df_prod.status_produto,
        ),
        df_itens.product_code == df_prod.product_id, "left"
    )
    .join(
        df_ped_gold.select(
            "id_pedido","data_pedido","mes_pedido","ano_pedido","trimestre",
            "status_pedido","id_cliente","id_vendedor",
            "canal_id","nome_canal","regional_code","regional_name",
            "segmento_cliente","porte_cliente"
        ),
        df_itens.order_id == df_ped_gold.id_pedido, "left"
    )
    .select(
        df_itens.order_id.alias("id_pedido"),
        df_itens.item_seq.alias("seq_item"),
        df_itens.product_code.alias("id_produto"),
        "nome_produto", "categoria", "subcategoria", "familia", "status_produto",
        df_itens.quantity.alias("quantidade"),
        df_itens.unit_price.alias("preco_unitario"),
        df_itens.total_item.alias("receita_item"),
        df_itens.item_status.alias("status_item"),
        "flag_total_divergente",
        "data_pedido", "mes_pedido", "ano_pedido", "trimestre",
        "status_pedido", "id_cliente", "id_vendedor",
        "canal_id", "nome_canal", "regional_code", "regional_name",
        "segmento_cliente", "porte_cliente",
    )
)

df_fct_itens.write.format("delta").mode("overwrite").saveAsTable("gold.fct_itens_pedido")
print(f"gold.fct_itens_pedido: {df_fct_itens.count()} registros")

gold.fct_itens_pedido: 1023 registros


## 8. gold.fct_entregas

Enriquecida com data_prometida do pedido. Calcula flag_atraso e dias_atraso.

In [0]:
df_ent     = spark.table("silver.fct_entregas")
df_ped_ref = spark.table("gold.fct_pedidos").select(
    "id_pedido","data_prometida","canal_id","nome_canal",
    "regional_code","regional_name","segmento_cliente"
)

df_fct_entregas = (
    df_ent
    .join(df_ped_ref, df_ent.order_id == df_ped_ref.id_pedido, "left")
    .withColumn("flag_atraso",
        F.when(
            (F.col("delivery_status_norm") == "Entregue") &
            (F.col("delivered_at").cast("date") > F.col("data_prometida")),
            True
        ).otherwise(False)
    )
    .withColumn("dias_atraso",
        F.when(F.col("flag_atraso"),
               F.datediff(F.col("delivered_at").cast("date"), F.col("data_prometida")))
         .otherwise(0)
    )
    .select(
        F.col("delivery_id").alias("id_entrega"),
        F.col("order_id").alias("id_pedido"),
        F.col("delivery_status_norm").alias("status_entrega"),
        F.col("carrier_name").alias("transportadora"),
        F.col("carrier_mode").alias("modal"),
        F.col("shipped_at").alias("data_envio"),
        F.col("delivered_at").alias("data_entrega"),
        "data_prometida",
        "dias_transito", "flag_atraso", "dias_atraso",
        F.col("shipping_cost").alias("custo_frete"),
        "flag_carrier_ausente",
        F.col("dest_state").alias("estado_destino"),
        F.col("dest_city").alias("cidade_destino"),
        "canal_id", "nome_canal", "regional_code", "regional_name", "segmento_cliente",
    )
)

df_fct_entregas.write.format("delta").mode("overwrite").saveAsTable("gold.fct_entregas")
print(f"gold.fct_entregas: {df_fct_entregas.count()} registros")

gold.fct_entregas: 331 registros


## 9. gold.fct_ocorrencias

In [0]:
df_oc      = spark.table("silver.fct_ocorrencias")
df_ped_ref2 = spark.table("gold.fct_pedidos").select(
    "id_pedido","data_pedido","mes_pedido","ano_pedido",
    "canal_id","nome_canal","regional_code","id_cliente",
    "segmento_cliente","status_pedido"
)

df_fct_oc = (
    df_oc
    .join(df_ped_ref2, df_oc.order_id == df_ped_ref2.id_pedido, "left")
    .select(
        F.col("ticket_id").alias("id_ticket"),
        F.col("order_id").alias("id_pedido"),
        F.col("event_type").alias("tipo_ocorrencia"),
        F.col("severity").alias("severidade"),
        F.col("status").alias("status_ticket"),
        F.col("created_at").alias("data_abertura"),
        F.date_format("created_at", "yyyy-MM").alias("mes_abertura"),
        F.year("created_at").alias("ano_abertura"),
        "id_cliente", "canal_id", "nome_canal",
        "regional_code", "segmento_cliente", "status_pedido",
    )
)

df_fct_oc.write.format("delta").mode("overwrite").saveAsTable("gold.fct_ocorrencias")
print(f"gold.fct_ocorrencias: {df_fct_oc.count()} registros")

gold.fct_ocorrencias: 277 registros


## 10. gold.mart_desempenho_mensal

KPIs pré-calculados por mês / canal / região / segmento. Permite dashboards sem joins adicionais.

In [0]:
df_ped = spark.table("gold.fct_pedidos")

df_mart = (
    df_ped
    .groupBy(
        "mes_pedido","ano_pedido","trimestre",
        "canal_id","nome_canal","tipo_canal",
        "regional_code","regional_name",
        "segmento_cliente","porte_cliente"
    )
    .agg(
        F.count("id_pedido").alias("total_pedidos"),
        F.countDistinct("id_cliente").alias("clientes_unicos"),
        F.sum("receita_bruta").alias("receita_bruta_total"),
        F.sum("desconto").alias("desconto_total"),
        F.sum("receita_liquida").alias("receita_liquida_total"),
        F.round(F.avg("receita_liquida"), 2).alias("ticket_medio"),
        F.sum(F.col("flag_cancelado").cast("int")).alias("pedidos_cancelados"),
        F.sum("qtd_itens").alias("itens_total"),
        F.round(F.avg("pct_desconto"), 2).alias("pct_desconto_medio"),
    )
    .withColumn("taxa_cancelamento_pct",
        F.round(F.col("pedidos_cancelados") / F.col("total_pedidos") * 100, 2)
    )
    .orderBy("mes_pedido","canal_id","regional_code")
)

df_mart.write.format("delta").mode("overwrite").saveAsTable("gold.mart_desempenho_mensal")
print(f"gold.mart_desempenho_mensal: {df_mart.count()} combinações")

gold.mart_desempenho_mensal: 376 combinações


## Validação Final — Gold Layer

In [0]:
gold_tables = [
    ("gold.dim_clientes",           "dim"),
    ("gold.dim_produtos",           "dim"),
    ("gold.dim_vendedores",         "dim"),
    ("gold.fct_pedidos",            "fct"),
    ("gold.fct_itens_pedido",       "fct"),
    ("gold.fct_entregas",           "fct"),
    ("gold.fct_ocorrencias",        "fct"),
    ("gold.mart_desempenho_mensal", "mart"),
]

print("=" * 60)
print("RESUMO — GOLD LAYER (sandbox.gold.*)")
print("=" * 60)
for t, tipo in gold_tables:
    count = spark.table(t).count()
    print(f"  [{tipo}] {t:<40} {count:>6} registros")

RESUMO — GOLD LAYER (sandbox.gold.*)
  [dim] gold.dim_clientes                           183 registros
  [dim] gold.dim_produtos                            71 registros
  [dim] gold.dim_vendedores                          40 registros
  [fct] gold.fct_pedidos                            413 registros
  [fct] gold.fct_itens_pedido                      1023 registros
  [fct] gold.fct_entregas                           331 registros
  [fct] gold.fct_ocorrencias                        277 registros
  [mart] gold.mart_desempenho_mensal                 376 registros


## Queries de Validação de Negócio

In [0]:
print("--- KPIs Globais ---")
spark.sql("""
    SELECT
        COUNT(*)                                        AS total_pedidos,
        ROUND(SUM(receita_liquida), 2)                  AS receita_liquida_total,
        ROUND(AVG(receita_liquida), 2)                  AS ticket_medio,
        ROUND(SUM(CAST(flag_cancelado AS INT)) * 100.0
              / COUNT(*), 2)                            AS taxa_cancelamento_pct,
        MIN(data_pedido)                                AS data_min,
        MAX(data_pedido)                                AS data_max
    FROM gold.fct_pedidos
""").show()

print("--- Receita por Canal ---")
spark.sql("""
    SELECT nome_canal, COUNT(*) pedidos,
           ROUND(SUM(receita_liquida),2) receita,
           ROUND(AVG(receita_liquida),2) ticket_medio
    FROM gold.fct_pedidos
    GROUP BY nome_canal ORDER BY receita DESC
""").show()

print("--- Receita por Região ---")
spark.sql("""
    SELECT regional_name, COUNT(*) pedidos,
           ROUND(SUM(receita_liquida),2) receita
    FROM gold.fct_pedidos
    GROUP BY regional_name ORDER BY receita DESC
""").show()

print("--- Top Categorias por Receita ---")
spark.sql("""
    SELECT categoria, ROUND(SUM(receita_item),2) receita, COUNT(*) itens
    FROM gold.fct_itens_pedido WHERE status_item = 'Ativo'
    GROUP BY categoria ORDER BY receita DESC
""").show()

print("--- Taxa de Atraso nas Entregas ---")
spark.sql("""
    SELECT COUNT(*) total,
           SUM(CAST(flag_atraso AS INT)) atrasadas,
           ROUND(SUM(CAST(flag_atraso AS INT))*100.0/COUNT(*),2) taxa_atraso_pct,
           ROUND(AVG(dias_atraso),1) dias_atraso_medio
    FROM gold.fct_entregas WHERE status_entrega = 'Entregue'
""").show()

print("--- Ocorrências por Tipo ---")
spark.sql("""
    SELECT tipo_ocorrencia, COUNT(*) total,
           SUM(CASE WHEN status_ticket='Aberto' THEN 1 ELSE 0 END) abertas
    FROM gold.fct_ocorrencias
    GROUP BY tipo_ocorrencia ORDER BY total DESC
""").show()

--- KPIs Globais ---
+-------------+---------------------+------------+---------------------+----------+----------+
|total_pedidos|receita_liquida_total|ticket_medio|taxa_cancelamento_pct|  data_min|  data_max|
+-------------+---------------------+------------+---------------------+----------+----------+
|          413|           2991218.15|     7242.66|                13.32|2025-01-04|2026-02-25|
+-------------+---------------------+------------+---------------------+----------+----------+

--- Receita por Canal ---
+------------+-------+---------+------------+
|  nome_canal|pedidos|  receita|ticket_medio|
+------------+-------+---------+------------+
|Inside Sales|    105| 796293.1|     7583.74|
|   Parceiros|     59|491715.75|     8334.17|
|  E-commerce|     63|479447.44|     7610.28|
| Marketplace|     57|365840.44|     6418.25|
|        NULL|     53|341400.08|     6441.51|
| Field Sales|     42|304008.44|      7238.3|
|    Revendas|     34| 212512.9|     6250.38|
+------------+---